In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import io
import os
from pathlib import Path
from bs4 import BeautifulSoup
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries loaded ✓')

Libraries loaded ✓


In [ ]:
# Authenticate and connect to Google Drive
from google.colab import auth
auth.authenticate_user()

from google.auth import default
creds, _ = default()
service = build('drive', 'v3', credentials=creds)

DRIVE_FOLDER_ID = '17Jcona7Ur-iV7miakhLkhHGBgggT0to7'  # food_data folder

print('Drive service ready ✓')

Drive service ready ✓


In [ ]:
# ============================================================
# LOAD ALL DATASETS FROM DRIVE
# ============================================================

def load_from_drive(filename):
    """Load a CSV from the project Drive folder."""
    results = service.files().list(
        q=f"name='{filename}' and '{DRIVE_FOLDER_ID}' in parents and trashed=false",
        fields="files(id, name)"
    ).execute()
    if not results['files']:
        print(f'⚠️  {filename} not found in Drive')
        return None
    file_id = results['files'][0]['id']
    request = service.files().get_media(fileId=file_id)
    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    fh.seek(0)
    return pd.read_csv(fh)

# Load all project datasets
nutrients_diaas    = load_from_drive('nutrients_diaas.csv')
nutrients_analysis = load_from_drive('nutrients_analysis.csv')
aa_merged          = load_from_drive('aa_merged.csv')

print(f'nutrients_diaas    : {nutrients_diaas.shape[0]} rows × {nutrients_diaas.shape[1]} cols')
print(f'nutrients_analysis : {nutrients_analysis.shape[0]} rows × {nutrients_analysis.shape[1]} cols')
print(f'aa_merged          : {aa_merged.shape[0]} rows × {aa_merged.shape[1]} cols')

print('\nnutrients_diaas columns:')
print(list(nutrients_diaas.columns))
print('\nnutrients_analysis columns:')
print(list(nutrients_analysis.columns))
print('\naa_merged columns:')
print(list(aa_merged.columns))

nutrients_diaas    : 31 rows × 19 cols
nutrients_analysis : 25 rows × 19 cols
aa_merged          : 26 rows × 52 cols

nutrients_diaas columns:
['food_key', 'source_category', 'fdc_id', 'description', 'usda_notes', 'protein_g', 'energy_kcal', 'fat_g', 'carbs_g', 'water_g', 'protein_pct_kcal', 'diaas_score', 'price_jpy', 'weight_g', 'price_jpy_per_100g', 'price_source', 'notes', 'yen_per_g_protein', 'adjusted_yen_per_g']

nutrients_analysis columns:
['food_key', 'source_category', 'fdc_id', 'description', 'usda_notes', 'protein_g', 'energy_kcal', 'fat_g', 'carbs_g', 'water_g', 'protein_pct_kcal', 'diaas_score', 'price_jpy', 'weight_g', 'price_jpy_per_100g', 'price_source', 'notes', 'yen_per_g_protein', 'adjusted_yen_per_g']

aa_merged columns:
['food_key', 'fdc_id', 'source_category', 'alanine', 'arginine', 'aspartic_acid', 'cystine', 'glutamic_acid', 'glycine', 'histidine', 'isoleucine', 'leucine', 'lysine', 'methionine', 'phenylalanine', 'proline', 'serine', 'threonine', 'tryptophan', 

In [ ]:
# ============================================================
# TABLEAU EXPORT — Notebook 05
# Three CSV files shaped for Tableau Public
# ============================================================

# First align aa_merged to 25 consolidated foods
DROP_LIST = ['pork_tonkatsu', 'pork_shougayaki', 'pork_shabushabu',
             'beef_usugiri', 'beef_loin', 'chicken_sasami']

aa_clean = aa_merged[~aa_merged['food_key'].isin(DROP_LIST)].copy().reset_index(drop=True)
print(f'aa_merged aligned: {len(aa_clean)} foods')

# Verify alignment
assert len(aa_clean) == len(nutrients_analysis), \
    f'Row mismatch: aa_clean={len(aa_clean)}, nutrients_analysis={len(nutrients_analysis)}'
print('✓ Row counts aligned')

aa_merged aligned: 25 foods
✓ Row counts aligned


In [ ]:
# ============================================================
# CSV 1: tableau_main.csv
# Wide format — one row per food, all core metrics
# Primary table for most Tableau views
# ============================================================

# Build pre/post DIAAS ranks
nutrients_analysis['pre_diaas_rank']  = nutrients_analysis['yen_per_g_protein'].rank().astype(int)
nutrients_analysis['post_diaas_rank'] = nutrients_analysis['adjusted_yen_per_g'].rank().astype(int)
nutrients_analysis['rank_change']     = nutrients_analysis['pre_diaas_rank'] - nutrients_analysis['post_diaas_rank']

# Merge with key AA metrics from aa_clean
aa_key_cols = [
    'food_key', 'leucine', 'total_bcaa_g', 'leucine_per_100yen',
    'bcaa_per_100yen', 'lysine', 'lysine_per_100yen',
    'saa_g', 'saa_vs_fao', 'saa_limiting',
    'g_food_for_threshold', 'cost_to_hit_threshold', 'protein_at_threshold',
    'bcaa_ratio_score', 'leu_ile_ratio', 'val_ile_ratio',
    'trp_vs_fao', 'trp_limiting',
    'is_complete_protein', 'min_iaa_ratio', 'limiting_aas'
]

tableau_main = nutrients_analysis.merge(
    aa_clean[aa_key_cols], on='food_key', how='left'
)

# Add muscle score
tableau_main['muscle_score'] = (
    tableau_main['leucine'] * 0.70 +
    (tableau_main['diaas_score'] / 100) * 0.30
).round(3)

tableau_main['muscle_cost'] = (
    tableau_main['price_jpy_per_100g'] / tableau_main['muscle_score']
).round(2)

tableau_main['muscle_rank'] = tableau_main['muscle_cost'].rank().astype(int)

# Clean up columns not needed in Tableau
DROP_COLS = ['fdc_id', 'usda_notes', 'price_jpy', 'weight_g']
tableau_main = tableau_main.drop(columns=DROP_COLS)

print(f'tableau_main: {tableau_main.shape[0]} rows × {tableau_main.shape[1]} columns')
print(f'\nColumns: {list(tableau_main.columns)}')

tableau_main: 25 rows × 41 columns

Columns: ['food_key', 'source_category', 'description', 'protein_g', 'energy_kcal', 'fat_g', 'carbs_g', 'water_g', 'protein_pct_kcal', 'diaas_score', 'price_jpy_per_100g', 'price_source', 'notes', 'yen_per_g_protein', 'adjusted_yen_per_g', 'pre_diaas_rank', 'post_diaas_rank', 'rank_change', 'leucine', 'total_bcaa_g', 'leucine_per_100yen', 'bcaa_per_100yen', 'lysine', 'lysine_per_100yen', 'saa_g', 'saa_vs_fao', 'saa_limiting', 'g_food_for_threshold', 'cost_to_hit_threshold', 'protein_at_threshold', 'bcaa_ratio_score', 'leu_ile_ratio', 'val_ile_ratio', 'trp_vs_fao', 'trp_limiting', 'is_complete_protein', 'min_iaa_ratio', 'limiting_aas', 'muscle_score', 'muscle_cost', 'muscle_rank']


In [ ]:
# ============================================================
# CSV 2: tableau_long_ranking.csv
# Long format — one row per food per metric
# For pre/post DIAAS comparison and ranking views
# ============================================================

ranking_rows = []
for _, row in tableau_main.iterrows():
    # Pre-DIAAS
    ranking_rows.append({
        'food_key':         row['food_key'],
        'source_category':  row['source_category'],
        'metric':           'Pre-DIAAS (raw)',
        'yen_per_g':        row['yen_per_g_protein'],
        'rank':             row['pre_diaas_rank'],
        'rank_change':      row['rank_change'],
        'diaas_score':      row['diaas_score'],
        'price_jpy_per_100g': row['price_jpy_per_100g'],
        'protein_g':        row['protein_g'],
    })
    # Post-DIAAS
    ranking_rows.append({
        'food_key':         row['food_key'],
        'source_category':  row['source_category'],
        'metric':           'Post-DIAAS (quality-adjusted)',
        'yen_per_g':        row['adjusted_yen_per_g'],
        'rank':             row['post_diaas_rank'],
        'rank_change':      row['rank_change'],
        'diaas_score':      row['diaas_score'],
        'price_jpy_per_100g': row['price_jpy_per_100g'],
        'protein_g':        row['protein_g'],
    })
    # Muscle score
    ranking_rows.append({
        'food_key':         row['food_key'],
        'source_category':  row['source_category'],
        'metric':           'Muscle cost efficiency',
        'yen_per_g':        row['muscle_cost'],
        'rank':             row['muscle_rank'],
        'rank_change':      row['pre_diaas_rank'] - row['muscle_rank'],
        'diaas_score':      row['diaas_score'],
        'price_jpy_per_100g': row['price_jpy_per_100g'],
        'protein_g':        row['protein_g'],
    })

tableau_long_ranking = pd.DataFrame(ranking_rows)
print(f'tableau_long_ranking: {tableau_long_ranking.shape[0]} rows × {tableau_long_ranking.shape[1]} columns')
print(f'Metrics: {tableau_long_ranking["metric"].unique().tolist()}')

# ============================================================
# CSV 3: tableau_aa_long.csv
# Long format — one row per food per amino acid
# For amino acid heatmap and radar charts in Tableau
# ============================================================

IAA_COLS = ['histidine', 'isoleucine', 'leucine', 'lysine',
            'methionine', 'phenylalanine', 'threonine', 'tryptophan', 'valine']

DISP_COLS = ['alanine', 'arginine', 'aspartic_acid', 'cystine',
             'glutamic_acid', 'glycine', 'proline', 'serine', 'tyrosine']

FAO_REFERENCE = {
    'histidine':     16,
    'isoleucine':    30,
    'leucine':       61,
    'lysine':        48,
    'methionine':    23,
    'phenylalanine': 41,
    'threonine':     25,
    'tryptophan':     6,
    'valine':        40,
}

aa_rows = []
for _, row in aa_clean.iterrows():
    for aa in IAA_COLS + DISP_COLS:
        if aa not in aa_clean.columns:
            continue
        val = row[aa]
        fao_ref  = FAO_REFERENCE.get(aa, None)
        fao_ratio = round(
            (val / row['protein_g'] * 1000) / fao_ref, 2
        ) if fao_ref and row['protein_g'] > 0 else None

        aa_rows.append({
            'food_key':        row['food_key'],
            'source_category': row['source_category'],
            'amino_acid':      aa,
            'aa_type':         'IAA' if aa in IAA_COLS else 'dispensable',
            'value_g':         round(val, 3) if pd.notna(val) else None,
            'fao_ratio':       fao_ratio,
            'fao_ref_mg_per_g': fao_ref,
            'is_limiting':     (fao_ratio < 1.0) if fao_ratio else None,
            'protein_g':       row['protein_g'],
            'price_jpy_per_100g': row['price_jpy_per_100g'],
            'diaas_score':     row['diaas_score'],
        })

tableau_aa_long = pd.DataFrame(aa_rows)
print(f'\ntableau_aa_long: {tableau_aa_long.shape[0]} rows × {tableau_aa_long.shape[1]} columns')
print(f'Amino acids: {tableau_aa_long["amino_acid"].unique().tolist()}')
print(f'Foods: {tableau_aa_long["food_key"].nunique()}')

tableau_long_ranking: 75 rows × 9 columns
Metrics: ['Pre-DIAAS (raw)', 'Post-DIAAS (quality-adjusted)', 'Muscle cost efficiency']

tableau_aa_long: 450 rows × 11 columns
Amino acids: ['histidine', 'isoleucine', 'leucine', 'lysine', 'methionine', 'phenylalanine', 'threonine', 'tryptophan', 'valine', 'alanine', 'arginine', 'aspartic_acid', 'cystine', 'glutamic_acid', 'glycine', 'proline', 'serine', 'tyrosine']
Foods: 25


In [ ]:
# ============================================================
# CSV 4: smart_shopper_long.csv
# Long format
# ============================================================

# Load smart_shopper from Drive
smart_shopper = load_from_drive('tableau_smart_shopper.csv')
print(f'Loaded: {smart_shopper.shape[0]} rows × {smart_shopper.shape[1]} columns')
print(smart_shopper.columns.tolist())

# Reshape smart shopper to long format for Tableau
ss_long_rows = []

threshold_map = {
    'beat_next_overall': 'Beat next overall',
    'beat_next_in_cat':  'Beat next in category',
    'beat_overall_#1':   'Beat overall #1 (egg)',
}

for _, row in smart_shopper.iterrows():
    for col, label in threshold_map.items():
        val = row[col]
        try:
            threshold = float(val)
        except:
            threshold = None

        ss_long_rows.append({
            'food_key':               row['food_key'],
            'source_category':        row['source_category'],
            'overall_rank':           row['overall_rank'],
            'current_price_per_100g': row['current_price_per_100g'],
            'threshold_type':         label,
            'threshold_price':        threshold,
            'status':                 val if threshold is None else None,
            'pct_drop_needed':        round((row['current_price_per_100g'] - threshold) / row['current_price_per_100g'] * 100, 1) if threshold else None,
        })

ss_long = pd.DataFrame(ss_long_rows)



Loaded: 25 rows × 14 columns
['food_key', 'source_category', 'current_price_per_100g', 'current_adjusted_¥/g', 'overall_rank', 'cat_rank', 'beat_next_overall', 'beat_next_overall_ref', 'beat_next_in_cat', 'beat_next_in_cat_ref', 'beat_cat_winner', 'beat_cat_winner_ref', 'beat_overall_#1', 'beat_overall_#1_ref']
✓ Uploaded tableau_smart_shopper_long.csv
Shape: 75 rows × 8 columns
         food_key source_category  overall_rank  current_price_per_100g        threshold_type  threshold_price                 status  pct_drop_needed
      chicken_egg         poultry             1                   53.20     Beat next overall              NaN       N/A — #1 overall              NaN
      chicken_egg         poultry             1                   53.20 Beat next in category              NaN N/A — is category best              NaN
      chicken_egg         poultry             1                   53.20 Beat overall #1 (egg)              NaN    N/A — is overall #1              NaN
   chicken_bre

In [ ]:
# ============================================================
# UPLOAD ALL THREE TABLEAU CSVs TO DRIVE
# ============================================================

# All exports
tableau_exports = [
    ('tableau_main.csv',                  tableau_main,         '/content/tableau_main.csv'),
    ('tableau_long_ranking.csv',          tableau_long_ranking, '/content/tableau_long_ranking.csv'),
    ('tableau_aa_long.csv',               tableau_aa_long,      '/content/tableau_aa_long.csv'),
    ('tableau_smart_shopper.csv',         smart_shopper,        '/content/tableau_smart_shopper.csv'),
    ('tableau_smart_shopper_long.csv',    ss_long,              '/content/tableau_smart_shopper_long.csv'),
]

for filename, dataframe, local_path in tableau_exports:
    dataframe.to_csv(local_path, index=False)
    print(f'Saved locally: {filename} ({dataframe.shape[0]} rows × {dataframe.shape[1]} cols)')

    existing = service.files().list(
        q=f"name='{filename}' and '{DRIVE_FOLDER_ID}' in parents and trashed=false",
        fields="files(id, name)"
    ).execute()
    for f in existing['files']:
        service.files().delete(fileId=f['id']).execute()
        print(f'  Deleted existing {filename}')

    media    = MediaFileUpload(local_path, mimetype='text/csv')
    meta     = {'name': filename, 'parents': [DRIVE_FOLDER_ID]}
    uploaded = service.files().create(body=meta, media_body=media, fields='id, name').execute()
    print(f'  ✓ Uploaded {uploaded["name"]} (id: {uploaded["id"]})\n')

print('All Tableau exports uploaded ✓')

Saved locally: tableau_main.csv (25 rows × 41 cols)
  Deleted existing tableau_main.csv
  ✓ Uploaded tableau_main.csv (id: 1bfpBC1IWoqPK89M-xXteAG_bDNpXYoDg)

Saved locally: tableau_long_ranking.csv (75 rows × 9 cols)
  Deleted existing tableau_long_ranking.csv
  ✓ Uploaded tableau_long_ranking.csv (id: 17J8nZEIN2HOJ39MT2ALDZukdKjT5iJEc)

Saved locally: tableau_aa_long.csv (450 rows × 11 cols)
  Deleted existing tableau_aa_long.csv
  ✓ Uploaded tableau_aa_long.csv (id: 14M11pcfjRra-CvSmCnCvdpWf7xweZ3km)

Saved locally: tableau_smart_shopper.csv (25 rows × 14 cols)
  Deleted existing tableau_smart_shopper.csv
  ✓ Uploaded tableau_smart_shopper.csv (id: 1j29uRkizJ5buK-6PKDv9dx7VN5dSjTNy)

Saved locally: tableau_smart_shopper_long.csv (75 rows × 8 cols)
  Deleted existing tableau_smart_shopper_long.csv
  ✓ Uploaded tableau_smart_shopper_long.csv (id: 1F8mXbzMVgSmVRrEuXEVG_7T00NDYcTZx)

All Tableau exports uploaded ✓


# Japan Protein Cost-Efficiency Analysis
## Notebook 05 — Tableau Data Preparation

**Goal:** Shape all analytical outputs into Tableau-ready CSV files for dashboard building.

---

## What this notebook produced

### Five Tableau-ready exports

| File | Format | Rows | Purpose |
|---|---|---|---|
| `tableau_main.csv` | Wide | 25 | Primary table — all metrics per food |
| `tableau_long_ranking.csv` | Long | 75 | Pre/post DIAAS and muscle ranking comparison |
| `tableau_aa_long.csv` | Long | 450 | Amino acid heatmap and IAA completeness |
| `tableau_smart_shopper.csv` | Wide | 25 | Smart Shopper — break-even thresholds per food |
| `tableau_smart_shopper_long.csv` | Long | 75 | Smart Shopper reshaped for Tableau bar chart |

### tableau_main.csv — key columns for Tableau
| Column | Use |
|---|---|
| `food_key`, `source_category` | Dimensions for filtering and color |
| `yen_per_g_protein` | Pre-DIAAS cost efficiency |
| `adjusted_yen_per_g` | Post-DIAAS cost efficiency |
| `pre_diaas_rank`, `post_diaas_rank`, `rank_change` | Ranking comparison |
| `muscle_cost`, `muscle_rank` | Bodybuilder-specific ranking |
| `leucine`, `leucine_per_100yen` | Muscle-building cost metric |
| `is_complete_protein`, `limiting_aas` | Protein quality flags |
| `g_food_for_threshold`, `cost_to_hit_threshold` | Leucine threshold analysis |
| `price_jpy_per_100g` | Raw price for smart shopper filters |

### tableau_long_ranking.csv — key columns
| Column | Use |
|---|---|
| `metric` | Filter: Pre-DIAAS / Post-DIAAS / Muscle cost |
| `yen_per_g` | Value axis for ranking charts |
| `rank` | Rank per metric |
| `rank_change` | Arrows / slope chart between metrics |

### tableau_aa_long.csv — key columns
| Column | Use |
|---|---|
| `amino_acid` | Column dimension for heatmap |
| `aa_type` | Filter: IAA vs dispensable |
| `value_g` | Absolute AA content |
| `fao_ratio` | Color encoding — red < 1.0, green > 1.0 |
| `is_limiting` | Boolean filter for highlighting |

### tableau_smart_shopper.csv — key columns
| Column | Use |
|---|---|
| `food_key`, `source_category` | Dimensions for filtering |
| `current_price_per_100g` | Current market price |
| `overall_rank` | Current quality-adjusted rank |
| `beat_next_overall` | Max price to beat next overall rank |
| `beat_next_in_cat` | Max price to beat next in category |
| `beat_overall_#1` | Max price to beat chicken egg |

### tableau_smart_shopper_long.csv — key columns
| Column | Use |
|---|---|
| `food_key` | Filter dimension — user selects food |
| `threshold_type` | Bar label: next overall / next in cat / overall #1 |
| `threshold_price` | Value axis — the buy signal price |
| `current_price_per_100g` | Reference line — current market price |
| `pct_drop_needed` | Label — % discount needed |
| `status` | Tooltip — N/A explanation where threshold doesn't apply |

---

## Consolidation applied in this project
The following duplicate entries were removed across Notebooks 04 and 05:

| Removed | Retained | Reason |
|---|---|---|
| pork_tonkatsu, pork_shougayaki, pork_shabushabu | pork_loin | Identical price, DIAAS, fdc_id |
| beef_usugiri, beef_loin | beef_american_cab, beef_wagyu_steak | Same price point |
| chicken_sasami | chicken_breast | Same price and fdc_id at this market |

Final dataset: **25 foods** across 6 categories.

---

## Next steps

| Step | Tool | Status |
|---|---|---|
| Build Dashboard 1 — Rankings | Tableau Public | ✅ complete |
| Build Dashboard 2 — Smart Shopper | Tableau Public | ⬜ next |
| Build Dashboard 3 — Muscle Analysis | Tableau Public | ⬜ |
| Document methodology | GitHub README | ⬜ |
| Set up repository | GitHub | ⬜ |
| Smart Shopper app prototype | Claude Artifacts / web | ⬜ post-dashboard |

---

## Project notebook summary

| Notebook | Purpose | Output |
|---|---|---|
| 01 — USDA EDA | Load and explore SR Legacy data | `nutrients_base.csv` |
| 02 — Price Data | Collect and standardize prices | `nutrients_prices.csv` |
| 03 — DIAAS | Apply protein quality scores | `nutrients_diaas.csv` |
| 03b — Amino Acids | Amino acid profile analysis | `aa_merged.csv` |
| 04 — Analysis | Deep analysis and Smart Shopper | `nutrients_analysis.csv` |
| 05 — Tableau Prep | Shape data for dashboard | `tableau_*.csv` (5 files) |